# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayak-D/FlyRank---Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I choose a logistic regression model because it is simple, interpretable, and a natural way to compare against the Week-4 baseline on the same signal set. This lane benefits from a transparent model that uses graded visibility, freshness, and CTR features without rewarding unnecessary complexity.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

candidate_paths = [
    Path.cwd() / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
]

path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        'Could not find data/raw/content_refresh_anonymized.csv.\n'
        'Expected it under the repo root or one level above the notebook folder.'
    )

print('Loading dataset from:', path)
df = pd.read_csv(path)
print('Rows in dataset:', len(df))
print('Known features:', [c for c in df.columns if c in ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate', 'sessions_90d', 'content_age_days', 'word_count', 'scroll_rate']])

## 2. Split design

I use a grouped holdout split by `client_id` so the model is evaluated on different pages than it was trained on. This is honest for refresh prediction because client-level page behavior and product context can vary, and we do not want the model to simply memorize a client’s typical decay pattern.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Define the outcome and candidate features.
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
feature_columns = [
    'impressions_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'engagement_rate',
    'sessions_90d',
    'content_age_days',
    'word_count',
    'scroll_rate',
]

available_features = [c for c in feature_columns if c in df.columns]
print('Using features:', available_features)

base = df.dropna(subset=available_features + ['client_id', 'trend_direction']).reset_index(drop=True)
print('Rows after dropping missing features:', len(base))

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(base, groups=base['client_id']))
train = base.iloc[train_idx].reset_index(drop=True)
test = base.iloc[test_idx].reset_index(drop=True)

print('Train rows:', len(train))
print('Test rows:', len(test))
print('Train declining rate:', train['is_declining_label'].mean())
print('Test declining rate:', test['is_declining_label'].mean())

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

position_bins = [0, 3, 6, 11, 21, 51, 10_000]
position_labels = ['1-2', '3-5', '6-10', '11-20', '21-50', '50+']

# Build baseline score with the same split.
train['position_bucket'] = pd.cut(train['avg_position'].fillna(999), bins=position_bins, labels=position_labels, right=False)
position_ctr = train.groupby('position_bucket', observed=True)['ctr'].median().rename('expected_ctr')

for frame in [train, test]:
    frame['position_bucket'] = pd.cut(frame['avg_position'].fillna(999), bins=position_bins, labels=position_labels, right=False)
    frame['expected_ctr'] = frame['position_bucket'].map(position_ctr)
    frame['visibility_score'] = np.log1p(frame['impressions_90d']) / np.log1p(train['impressions_90d']).max()
    frame['staleness_score'] = frame['days_since_last_update'].clip(lower=0, upper=365) / 365
    frame['position_score'] = np.where(frame['avg_position'] > 0, ((50 - frame['avg_position'].clip(upper=50)) / 50), 0.0)
    frame['ctr_opportunity_score'] = np.where(
        (frame['avg_position'] <= 20)
        & (frame['ctr'] >= 0)
        & (frame['ctr'] < frame['expected_ctr'] - 0.01),
        ((frame['expected_ctr'] - frame['ctr']) / frame['expected_ctr']).clip(0, 1),
        0.0,
    )
    frame['baseline_score'] = (
        0.35 * frame['visibility_score']
        + 0.30 * frame['staleness_score']
        + 0.25 * frame['position_score']
        + 0.10 * frame['ctr_opportunity_score']
    ).clip(0, 1)

# Train a transparent model.
X_train = train[available_features]
y_train = train['is_declining_label']
X_test = test[available_features]
y_test = test['is_declining_label']

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
model.fit(X_train, y_train)

test['model_score'] = model.predict_proba(X_test)[:, 1]

metrics = []
for name, score_column in [('Baseline', 'baseline_score'), ('Logistic Regression', 'model_score')]:
    score = test[score_column]
    auc = roc_auc_score(y_test, score)
    top50 = test.nlargest(50, score_column)
    metrics.append({
        'method': name,
        'auc': float(auc),
        'top_50_declining_rate': float(top50['is_declining_label'].mean()),
    })

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

coef_df = pd.DataFrame({
    'feature': available_features,
    'coefficient': model.named_steps['logisticregression'].coef_[0],
}).sort_values('coefficient', ascending=False)
print('\nModel coefficients:')
print(coef_df.to_string(index=False))

## 4. Errors and interpretation

I inspect where the model disagrees with the baseline and the label, and I pay special attention to the top-ranked pages. This helps reveal whether the model is learning a real decline signal or simply ranking by visibility.

In [ ]:
# Compare top-ranked pages and error patterns.
comparison = test[['content_id', 'client_id', 'is_declining_label', 'baseline_score', 'model_score', 'avg_position', 'ctr', 'impressions_90d', 'days_since_last_update']].copy()
comparison['baseline_rank'] = comparison['baseline_score'].rank(method='first', ascending=False)
comparison['model_rank'] = comparison['model_score'].rank(method='first', ascending=False)
comparison['rank_diff'] = comparison['baseline_rank'] - comparison['model_rank']

print('Top 10 model predictions:')
print(comparison.nsmallest(10, 'model_rank')[['content_id', 'model_rank', 'model_score', 'baseline_score', 'is_declining_label', 'avg_position', 'ctr', 'impressions_90d']].to_string(index=False))

print('\nTop 10 baseline-only picks where the model does not agree:')
print(comparison.sort_values(['baseline_rank', 'model_rank']).head(10)[['content_id', 'baseline_rank', 'model_rank', 'is_declining_label', 'avg_position', 'ctr', 'impressions_90d']].to_string(index=False))

comparison['model_error'] = np.abs(comparison['model_score'] - comparison['is_declining_label'])
print('\nWorst model errors by absolute distance:')
print(comparison.nlargest(10, 'model_error')[['content_id', 'model_score', 'is_declining_label', 'avg_position', 'ctr', 'impressions_90d', 'days_since_last_update']].to_string(index=False))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.